# Naive RAG — pełny kontekst CSV

Najprostsze możliwe podejście: bierzemy pierwsze pytanie z `golden_questions.json` i wysyłamy je do modelu razem z **całym** `products_zurada.csv` wklejonym do kontekstu.  
Brak retrieval, brak embeddingów — czysty brute-force.

In [2]:
%pip install openai pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI

# ── Konfiguracja ───────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "Brak klucza")
MODEL              = "openai/gpt-5.4-mini"   # model Azure OpenAI przez OpenRouter
BASE_URL           = "https://openrouter.ai/api/v1"

BASE_DIR       = Path(".")  # lessons/3_naive_rag/
QUESTIONS_FILE = BASE_DIR / "golden_questions.json"
PRODUCTS_CSV   = BASE_DIR / "products_zurada.csv"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=BASE_URL,
)
print(f"Model: {MODEL}")
print(f"Klucz API: {'OK' if OPENROUTER_API_KEY != 'Brak klucza' else 'BRAK — ustaw OPENROUTER_API_KEY'}")

Model: openai/gpt-5.4-mini
Klucz API: OK


In [4]:
# ── Wczytaj pytanie i katalog produktów ────────────────────────────────────────
with open(QUESTIONS_FILE, encoding="utf-8") as f:
    questions = json.load(f)

question_obj = questions[0]
question     = question_obj["question"]

df = pd.read_csv(PRODUCTS_CSV)

print(f"Pytanie ({question_obj['id']}): {question}")
print(f"\nOczekiwane ID produktów: {question_obj['expected_product_ids']}")
print(f"\nKatalog: {len(df)} produktów, {len(df.columns)} kolumn")
print(f"Kolumny: {list(df.columns)}")

Pytanie (1): Muszę odtłuścić aluminiowe elementy w warsztacie. Większość środków odtłuszczających jest niedozwolona na aluminium — który produkt Zurada mogę bezpiecznie zastosować?

Oczekiwane ID produktów: [45]

Katalog: 100 produktów, 8 kolumn
Kolumny: ['product_id', 'product_name', 'title', 'content_encoded', 'short_description', 'right_description', 'technical_sheet', 'categories']


In [5]:
# ── Buduj kontekst — pełny CSV jako tekst ──────────────────────────────────────
catalog_text = df.to_csv(index=False)

# Szacunkowa liczba znaków
chars = len(catalog_text)
approx_tokens = chars // 4  # przybliżenie: ~4 znaki = 1 token
print(f"Rozmiar kontekstu CSV: {chars:,} znaków (~{approx_tokens:,} tokenów)")

Rozmiar kontekstu CSV: 121,906 znaków (~30,476 tokenów)


In [6]:
# ── Prompt systemowy i wywołanie API ───────────────────────────────────────────
SYSTEM_PROMPT = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Pomagasz klientom dobrać odpowiedni produkt do ich potrzeb.

Poniżej otrzymasz pełny katalog produktów Zurada w formacie CSV.
Na jego podstawie odpowiedz na pytanie klienta.

Zasady:
- Odpowiadaj wyłącznie na podstawie danych z katalogu.
- Podaj nazwę produktu i jego ID (kolumna product_id).
- Jeśli produkt pasuje — wytłumacz DLACZEGO, powołując się na konkretne właściwości.
- Jeśli żaden produkt nie pasuje — powiedz to wprost.
- Nie wymyślaj cech produktów spoza katalogu.
- Odpowiadaj po polsku."""

USER_MESSAGE = f"""Katalog produktów Zurada (CSV):

{catalog_text}

---

Pytanie klienta: {question}"""

print("Wysyłam zapytanie do modelu...")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_MESSAGE},
    ],
    max_completion_tokens=1000,
)

answer = response.choices[0].message.content
print("OK")

Wysyłam zapytanie do modelu...
OK


In [7]:
# ── Wynik ──────────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"PYTANIE: {question}")
print("=" * 60)
print()
print("ODPOWIEDŹ MODELU:")
print(answer)
print()
print("-" * 60)
print(f"Oczekiwane product_id: {question_obj['expected_product_ids']}")
print()
print("ASERCJE DO SPRAWDZENIA:")
for a in question_obj["assertions"]:
    print(f"  [ ] {a}")
print()
print(f"Tokeny: prompt={response.usage.prompt_tokens}, odpowiedź={response.usage.completion_tokens}, razem={response.usage.total_tokens}")

PYTANIE: Muszę odtłuścić aluminiowe elementy w warsztacie. Większość środków odtłuszczających jest niedozwolona na aluminium — który produkt Zurada mogę bezpiecznie zastosować?

ODPOWIEDŹ MODELU:
Do tego celu pasuje **Zurada Odtłuszczacz Moc (product_id: 45)**.

Dlaczego:
- w opisie ma wprost, że **można go stosować do powierzchni wrażliwych na działanie alkaliów, np. do aluminium**,
- jest przeznaczony do **odtłuszczania powierzchni i przedmiotów**,
- usuwa **zabrudzenia ropopochodne, oleje, smary oraz tłuszcze kuchenne**,
- nadaje się do **mycia ręcznego, maszynowego i ciśnieniowego**, więc sprawdzi się w warsztacie.

Jeśli chcesz, mogę też wskazać, którego produktu **nie** wybierać do aluminium z katalogu.

------------------------------------------------------------
Oczekiwane product_id: [45]

ASERCJE DO SPRAWDZENIA:
  [ ] Odpowiedź powinna wskazać produkt Zurada Odtłuszczacz Moc (ID 45)
  [ ] Odpowiedź powinna explicite potwierdzić, że produkt MOŻNA stosować na aluminium
  [ ] Od

In [8]:
# ── Funkcja ewaluacji — zwraca JSON z product_ids ──────────────────────────────
import time

SYSTEM_EVAL = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Poniżej otrzymasz pełny katalog produktów Zurada w formacie CSV.
Na jego podstawie odpowiedz na pytanie klienta.

Zasady:
- Odpowiadaj wyłącznie na podstawie danych z katalogu.
- Nie wymyślaj cech produktów spoza katalogu.
- Odpowiadaj po polsku.

Zwróć odpowiedź jako JSON:
{\"product_ids\": [lista rekomendowanych product_id jako liczby całkowite],
 \"answer\": \"krótkie uzasadnienie po polsku\"}"""

def ask_model(question: str) -> tuple[list[int], str, dict]:
    """Wysyła pytanie do modelu. Zwraca (product_ids, answer, usage)."""
    user_msg = f"""Katalog produktów Zurada (CSV):\n\n{catalog_text}\n\n---\n\nPytanie klienta: {question}"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_EVAL},
            {"role": "user",   "content": user_msg},
        ],
        response_format={"type": "json_object"},
        max_completion_tokens=400,
    )
    raw = resp.choices[0].message.content or "{}"
    data = json.loads(raw)
    ids = [int(x) for x in data.get("product_ids", [])]
    return ids, data.get("answer", ""), resp.usage

print("Funkcja ask_model gotowa.")

Funkcja ask_model gotowa.


In [10]:
# ── Pętla przez wszystkie pytania — expected vs returned product_id ────────────
SLEEP_BETWEEN = 0.1   # sekund między requestami

results = []
total_tokens = 0

for i, q in enumerate(questions):
    qid        = q["id"]
    question   = q["question"]
    expected   = set(q["expected_product_ids"])
    no_product = len(expected) == 0  # pytanie gdzie poprawna odpowiedź to 'brak produktu'

    try:
        returned_ids, answer, usage = ask_model(question)
        returned = set(returned_ids)
        total_tokens += usage.total_tokens
    except Exception as e:
        print(f"  [BŁĄD] pytanie {qid}: {e}")
        returned, answer = set(), str(e)

    hit    = expected & returned
    missed = expected - returned
    extra  = returned - expected

    if no_product:
        # Prawidłowa odpowiedź: model NIE powinien zwracać żadnego ID
        correct = len(returned) == 0
    else:
        correct = len(missed) == 0 and len(returned) > 0

    results.append({
        "id":         qid,
        "no_product": no_product,
        "expected":   sorted(expected),
        "returned":   sorted(returned),
        "hit":        sorted(hit),
        "missed":     sorted(missed),
        "extra":      sorted(extra),
        "correct":    correct,
        "answer":     answer[:120],
    })

    tag    = "[∅]" if no_product else "   "
    status = "✓" if correct else "✗"
    print(f"[{status}]{tag} Q{qid:02d} oczekiwane={sorted(expected) or '∅'} zwrócone={sorted(returned) or '∅'}")
    time.sleep(SLEEP_BETWEEN)

print(f"\nŁączne tokeny: {total_tokens:,}")


[✓]    Q01 oczekiwane=[45] zwrócone=[45]
[✓]    Q02 oczekiwane=[37] zwrócone=[37, 67]
[✓]    Q03 oczekiwane=[56] zwrócone=[56]
[✓]    Q04 oczekiwane=[16] zwrócone=[16]
[✓]    Q05 oczekiwane=[73] zwrócone=[73]
[✓]    Q06 oczekiwane=[49] zwrócone=[49]
[✓]    Q07 oczekiwane=[53] zwrócone=[53]
[✓]    Q08 oczekiwane=[65] zwrócone=[65]
[✓]    Q09 oczekiwane=[52] zwrócone=[52]
[✓]    Q10 oczekiwane=[64] zwrócone=[64]
[✗]    Q11 oczekiwane=[51] zwrócone=[50, 73]
[✓]    Q12 oczekiwane=[81] zwrócone=[81]
[✓]    Q13 oczekiwane=[86] zwrócone=[86]
[✓]    Q14 oczekiwane=[95] zwrócone=[95]
[✓]    Q15 oczekiwane=[28] zwrócone=[28, 30]
[✓]    Q16 oczekiwane=[17, 19] zwrócone=[17, 19]
[✓]    Q17 oczekiwane=[70] zwrócone=[70]
[✓]    Q18 oczekiwane=[15] zwrócone=[15]
[✓]    Q19 oczekiwane=[31] zwrócone=[31]
[✓]    Q20 oczekiwane=[87] zwrócone=[87]
[✓]    Q21 oczekiwane=[41] zwrócone=[41]
[✓]    Q22 oczekiwane=[54] zwrócone=[54]
[✓]    Q23 oczekiwane=[88] zwrócone=[88]
[✓]    Q24 oczekiwane=[55] zwrócone=[

In [11]:
# ── Podsumowanie wyników ───────────────────────────────────────────────────────
results_df = pd.DataFrame(results)

n_total      = len(results_df)
n_correct    = results_df["correct"].sum()
accuracy     = n_correct / n_total * 100

normal_df    = results_df[~results_df["no_product"]]
noproduct_df = results_df[results_df["no_product"]]

print("=" * 60)
print(f"WYNIK OGÓLNY: {n_correct}/{n_total} poprawnych ({accuracy:.1f}%)")
print(f"  Pytania z produktem:      {normal_df['correct'].sum()}/{len(normal_df)}")
print(f"  Pytania 'brak produktu':  {noproduct_df['correct'].sum()}/{len(noproduct_df)}")
print("=" * 60)
print()

display_cols = ["id", "no_product", "expected", "returned", "correct", "missed", "extra"]
print(results_df[display_cols].to_string(index=False))
print()

errors = results_df[~results_df["correct"]]
if not errors.empty:
    print(f"BŁĘDY ({len(errors)} pytań):")
    for _, row in errors.iterrows():
        tag = " [∅ model powinien odmówić]" if row["no_product"] else ""
        print(f"  Q{row['id']:02d}{tag}")
        print(f"    oczekiwane: {row['expected'] or '∅'} | zwrócone: {row['returned'] or '∅'}")
        print(f"    odpowiedź: {row['answer']}")


WYNIK OGÓLNY: 41/50 poprawnych (82.0%)
  Pytania z produktem:      40/46
  Pytania 'brak produktu':  1/4

 id  no_product expected returned  correct missed    extra
  1       False     [45]     [45]     True     []       []
  2       False     [37] [37, 67]     True     []     [67]
  3       False     [56]     [56]     True     []       []
  4       False     [16]     [16]     True     []       []
  5       False     [73]     [73]     True     []       []
  6       False     [49]     [49]     True     []       []
  7       False     [53]     [53]     True     []       []
  8       False     [65]     [65]     True     []       []
  9       False     [52]     [52]     True     []       []
 10       False     [64]     [64]     True     []       []
 11       False     [51] [50, 73]    False   [51] [50, 73]
 12       False     [81]     [81]     True     []       []
 13       False     [86]     [86]     True     []       []
 14       False     [95]     [95]     True     []       []
 15      

In [18]:
# ── Prompt systemowy i wywołanie API ───────────────────────────────────────────
SYSTEM_PROMPT = """Jesteś ekspertem ds. środków czystości firmy Zurada.
Pomagasz klientom dobrać odpowiedni produkt do ich potrzeb.

Poniżej otrzymasz pełny katalog produktów Zurada w formacie CSV.
Na jego podstawie odpowiedz na pytanie klienta.

Zasady:
- Odpowiadaj wyłącznie na podstawie danych z katalogu.
- Podaj nazwę produktu i jego ID (kolumna product_id).
- Jeśli produkt pasuje — wytłumacz DLACZEGO, powołując się na konkretne właściwości.
- Jeśli żaden produkt nie pasuje — powiedz to wprost.
- Nie wymyślaj cech produktów spoza katalogu.
- Odpowiadaj po polsku."""

USER_MESSAGE = f"""Katalog produktów Zurada (CSV):

{catalog_text}

---

Pytanie klienta: Woda w restauracji jest bardzo twarda, zmywarka zostawia przebarwienia po herbacie i kawie na szklankach. Potrzebuję preparatu do zmywarki gastronomicznej, który jest bezfosforanowy, wybiela i działa w twardej wodzie."""

print("Wysyłam zapytanie do modelu...")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_MESSAGE},
    ],
    max_completion_tokens=1000,
)

answer = response.choices[0].message.content
print(answer)

Wysyłam zapytanie do modelu...
Polecam produkt "Zurada Zmywark Pro" (product_id: 19), który jest bezfosforanowym preparatem przeznaczonym do maszynowego mycia i wybielania naczyń w zmywarkach gastronomicznych. Skutecznie usuwa ślady po herbacie i kawie, a także inne trudne do usunięcia zabrudzenia z tłuszczu, białka i skrobi. Działa skutecznie także w twardej wodzie, co jest ważne w Twoim przypadku. Produkt zawiera aktywny utleniacz, który wspiera proces wybielania.
